<a href="https://colab.research.google.com/github/BirasaDivine/sdg3-indicator-text-classification-/blob/main/test_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
from scipy.sparse import load_npz
import pandas as pd
from scipy.sparse import save_npz
import matplotlib.pyplot as plt
import joblib
from scipy.sparse import hstack
import seaborn as sns
from sklearn.metrics import hamming_loss, f1_score, precision_score, recall_score
import warnings
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('Ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/SDG_Assignment/processed_data'

# Labels
y_train = np.load(f'{SAVE_DIR}/y_train.npy')
y_val   = np.load(f'{SAVE_DIR}/y_val.npy')

# TF-IDF word features
X_train_tfidf = load_npz(f'{SAVE_DIR}/X_train_tfidf.npz')
X_val_tfidf   = load_npz(f'{SAVE_DIR}/X_val_tfidf.npz')

# Combined TF-IDF (word + char)
X_train_combined = load_npz(f'{SAVE_DIR}/X_train_combined_tfidf.npz')
X_val_combined   = load_npz(f'{SAVE_DIR}/X_val_combined_tfidf.npz')

# Indicator names
ALL_INDICATORS = np.load(f'{SAVE_DIR}/indicator_names.npy', allow_pickle=True).tolist()
SHORT_NAMES    = [ind.split(' - ')[0] if ' - ' in ind else ind for ind in ALL_INDICATORS]

# Label support (how many positive samples per label)
label_support = y_train.sum(axis=0)

print(f' Data loaded successfully')
print(f'   Train samples          : {y_train.shape[0]}')
print(f'   Val samples            : {y_val.shape[0]}')
print(f'   Number of labels       : {y_train.shape[1]}')
print(f'   TF-IDF word features   : {X_train_tfidf.shape[1]:,}')
print(f'   Combined TF-IDF feats  : {X_train_combined.shape[1]:,}')

 Data loaded successfully
   Train samples          : 2545
   Val samples            : 450
   Number of labels       : 27
   TF-IDF word features   : 50,000
   Combined TF-IDF feats  : 80,000


In [ ]:
df_test = pd.read_csv('/content/Devex_test_questions (1).csv', encoding='latin-1')

print('Test shape:', df_test.shape)
print('Columns:', df_test.columns.tolist())
print(f'Document types in test:\n{df_test["Type"].value_counts()}')

Test shape: (998, 3)
Columns: ['Unique ID', 'Type', 'Text']
Document types in test:
Type
Grant           275
Tender          233
Contract        177
Open Opp         90
News             71
Funding Info     60
Organization     59
Program          33
Name: count, dtype: int64


In [ ]:
tfidf_word = joblib.load(f'{SAVE_DIR}/tfidf_word_vectorizer.pkl')
tfidf_char = joblib.load(f'{SAVE_DIR}/tfidf_char_vectorizer.pkl')

print('Vectorizers loaded')

Vectorizers loaded


In [ ]:
for pkg in ['stopwords','wordnet','punkt','punkt_tab','omw-1.4']:
    nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()

SDG_KEEP = {
    'health','mortality','maternal','neonatal','disease','hiv','aids',
    'malaria','tuberculosis','mental','substance','vaccine','water',
    'sanitation','hygiene','nutrition','sexual','reproductive','death',
    'birth','child','infant','tobacco','alcohol','injury','universal'
}
STOP_WORDS = set(stopwords.words('english')) - SDG_KEEP

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b\d{4}\b', ' year ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def lemmatize_text(text):
    text   = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

print('Cleaning functions ready')

Cleaning functions ready


In [ ]:
print('Preprocessing test text')
X_test_lemma = [lemmatize_text(t) for t in df_test['Text'].values]

X_test_tfidf    = tfidf_word.transform(X_test_lemma)
X_test_char     = tfidf_char.transform(X_test_lemma)
X_test_combined = hstack([X_test_tfidf, X_test_char])

print(f'\n Test preprocessing complete')
print(f'X_test_tfidf shape   : {X_test_tfidf.shape}')
print(f'X_test_combined shape: {X_test_combined.shape}')

Preprocessing test text

 Test preprocessing complete
X_test_tfidf shape   : (998, 50000)
X_test_combined shape: (998, 80000)


In [ ]:
# Test features
save_npz(f'{SAVE_DIR}/X_test_tfidf.npz',    X_test_tfidf)
save_npz(f'{SAVE_DIR}/X_test_char.npz',     X_test_char)
save_npz(f'{SAVE_DIR}/X_test_combined.npz', X_test_combined)

np.save(f'{SAVE_DIR}/X_test_text.npy', np.array(X_test_lemma))


np.save(f'{SAVE_DIR}/test_ids.npy', df_test['Unique ID'].values)

print('Test data saved to Drive')

Test data saved to Drive


In [ ]:
import os

for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / (1024*1024)
    print(f'  {f:<45} {size:.1f} MB')

  X_test_char.npz                               12.0 MB
  X_test_combined.npz                           13.7 MB
  X_test_text.npy                               79.4 MB
  X_test_tfidf.npz                              1.7 MB
  X_train_char.npz                              31.2 MB
  X_train_combined_tfidf.npz                    36.7 MB
  X_train_final.npz                             9.2 MB
  X_train_sbert.npy                             3.7 MB
  X_train_text.npy                              6.1 MB
  X_train_tfidf.npz                             4.6 MB
  X_val_char.npz                                5.3 MB
  X_val_combined_tfidf.npz                      6.1 MB
  X_val_final.npz                               1.5 MB
  X_val_sbert.npy                               0.7 MB
  X_val_text.npy                                1.1 MB
  X_val_tfidf.npz                               0.7 MB
  indicator_names.npy                           0.0 MB
  test_ids.npy                                  0.0 MB
  tfi